# Stages and aggregations

## Overview

This page covers repeating part of a computation over many values, such as a list of files.
[Parameter Tables](parameter-tables.ipynb) covers the existing `map`/`reduce` mechanism for the same purpose.

A workflow often has a part that runs once, such as loading a calibration or building a mask, and a part that runs per input: per file, per detector bank, or per chunk of a stream.
The results of the per-input part are then combined, and the rest of the workflow runs once on the combined value.

Sciline expresses this with [Stage](../generated/classes/sciline.Stage.rst) and [Aggregation](../generated/classes/sciline.Aggregation.rst).
Both are built from an ordinary flat pipeline.
Nothing is added to the pipeline itself.

## A stage

We start with a pipeline similar to the one in [Getting Started](getting-started.ipynb).
Loading the calibration stands in for an expensive step that does not depend on the file.
The provider appends to a list so we can count how often it runs:

In [ ]:
import math
import operator
from typing import NewType

import sciline

_fake_filesystem = {
    'file102.txt': [1.0, 2.0, float('nan'), 3.0],
    'file103.txt': [1.0, 2.0, 3.0, 4.0],
    'file104.txt': [1.0, 2.0, 3.0, 4.0, 5.0],
    'file105.txt': [1.0, 2.0, 3.0],
    'calib_2025.txt': [0.5],
    'calib_2026.txt': [0.25],
}

# 1. Define domain types

Filename = NewType('Filename', str)
RawData = NewType('RawData', list)
CalibrationFilename = NewType('CalibrationFilename', str)
Calibration = NewType('Calibration', float)
Calibrated = NewType('Calibrated', list)
Total = NewType('Total', float)
ScaleFactor = NewType('ScaleFactor', float)
Result = NewType('Result', float)

# 2. Define providers

calibration_loads = []


def load(filename: Filename) -> RawData:
    """Load the data from a file."""
    return RawData(_fake_filesystem[filename])


def load_calibration(filename: CalibrationFilename) -> Calibration:
    """Load the calibration factor."""
    calibration_loads.append(filename)
    return Calibration(_fake_filesystem[filename][0])


def calibrate(raw: RawData, calibration: Calibration) -> Calibrated:
    """Drop NaNs and apply the calibration."""
    return Calibrated([x * calibration for x in raw if not math.isnan(x)])


def total(data: Calibrated) -> Total:
    """Sum the calibrated data."""
    return Total(sum(data))


def scale(total: Total, factor: ScaleFactor) -> Result:
    """Scale the total."""
    return Result(total * factor)


# 3. Create pipeline

providers = [load, load_calibration, calibrate, total, scale]
params = {CalibrationFilename: 'calib_2025.txt', ScaleFactor: 2.0}
pipeline = sciline.Pipeline(providers, params=params)
pipeline.visualize(Result, graph_attr={'rankdir': 'LR'})

`Filename` has no value.
A [Stage](../generated/classes/sciline.Stage.rst) cuts the pipeline at a set of *inputs*, keys supplied on each call, and computes a set of *outputs* from them.
Everything the outputs need that does not depend on the inputs is computed once, on first use, and held:

In [ ]:
stage = sciline.Stage(pipeline, outputs=(Result,), inputs=(Filename,))
{f: stage({Filename: f})[Result] for f in ['file102.txt', 'file103.txt', 'file104.txt']}

The stage was called three times, but the calibration was loaded once:

In [ ]:
calibration_loads

The stage tells us where it cut the pipeline.
`frontier` lists the held keys that the per-call part reads, `dynamic` lists the keys that depend on the inputs and are computed on each call, and `keys` lists every key the stage uses, held or per-call:

In [ ]:
stage.frontier

In [ ]:
stage.dynamic

In [ ]:
stage.keys

An input does not have to be a parameter.
If it is an intermediate result, its provider and everything upstream of it are cut off, and the stage computes only what lies downstream.
Here `Total` is supplied, so nothing is loaded and only `ScaleFactor` is held; the cut-off keys are not in `keys` either:

In [ ]:
from_total = sciline.Stage(pipeline, outputs=(Result,), inputs=(Total,))
from_total.frontier, from_total.dynamic, from_total.keys

In [ ]:
from_total({Total: 10.0})

A stage is a snapshot of the pipeline at the time it is built.
Set all parameters, except the inputs, before building it; changing the pipeline afterwards does not affect the stage.

## Warming several stages

Two stages built from the same pipeline may share held work.
Each stage computes its held part on first use, so used independently they would both load the calibration.
[warm](../generated/functions/sciline.warm.rst) computes the held parts of several stages in one run, so shared work is done once:

In [ ]:
calibration_loads.clear()
calibrated = sciline.Stage(pipeline, outputs=(Calibrated,), inputs=(Filename,))
totals = sciline.Stage(pipeline, outputs=(Total,), inputs=(Filename,))
sciline.warm(calibrated, totals)
calibration_loads

Each stage keeps only the values at its own frontier, available as `static`:

In [ ]:
calibrated.static, totals.static

## An aggregation

An [Aggregation](../generated/classes/sciline.Aggregation.rst) computes a contribution from each row of a table, combines the contributions, and computes the outputs from the combined values.
The columns of the table are the *member keys*, the rows are the *members*.
The keys at which contributions are combined are the *accumulation keys*; each has an accumulator that combines the values pushed to it.

Here we combine `Calibrated` by concatenation.
[Buffered](../generated/classes/sciline.Buffered.rst) makes accumulators that hold every pushed value and apply a function to all of them when the combined value is read:

In [ ]:
def concat(*parts: list) -> list:
    return [x for part in parts for x in part]


agg = sciline.Aggregation(
    pipeline,
    members=(Filename,),
    accumulators={Calibrated: sciline.Buffered(concat)},
    outputs=(Result,),
)

The table is a mapping from a label to a row, and a row is a mapping from member key to value:

In [ ]:
table = {
    102: {Filename: 'file102.txt'},
    103: {Filename: 'file103.txt'},
    104: {Filename: 'file104.txt'},
    105: {Filename: 'file105.txt'},
}
agg.compute(table)

Per member, the aggregation computed `Calibrated` from `Filename`.
The four lists were concatenated, and `Total` and `Result` were computed once from the concatenated list.
`accumulation_keys` lists the keys that are accumulated:

In [ ]:
agg.accumulation_keys

## Accumulators

An accumulator is any object with a `push` method and a `value` property, see [Accumulator](../generated/classes/sciline.Accumulator.rst).
The `accumulators` argument maps each accumulation key to a factory that returns a new, empty accumulator, so a class with a no-argument constructor can be passed directly.

`Buffered` holds every pushed value.
For a large dense sum over many members, a running total is much cheaper.
[Reduced](../generated/classes/sciline.Reduced.rst) makes accumulators that apply a binary function to the result so far and each pushed value, and hold only the result.
The function must be associative and must not modify its arguments.
Here we move the accumulation key from `Calibrated` to `Total` and sum as we go:

In [ ]:
agg = sciline.Aggregation(
    pipeline,
    members=(Filename,),
    accumulators={Total: sciline.Reduced(operator.add)},
    outputs=(Result,),
)
agg.compute(table)

The result is the same, but only one number per member is pushed, and the accumulator holds only the running total.
`compute` pushes each contribution as soon as it is made, before computing the next, so peak memory is the accumulator's choice.

## The three steps separately

`compute` is a loop over the table that calls three methods, which can also be called directly.
`contribute` computes the accumulation keys for one row, `combine` pushes contributions into new accumulators and reads them, and `finalize` computes the outputs from the combined values:

In [ ]:
contributions = {label: agg.contribute(row) for label, row in table.items()}
contributions

In [ ]:
combined = agg.combine(contributions.values())
combined

In [ ]:
agg.finalize(combined)

The aggregation holds its stages, not the contributions.
The three steps can therefore run in separate processes, with the contributions serialized between them.
A caller may also hold the contributions by label, as above, to add or remove members without recomputing the rest.
Here we drop one member:

In [ ]:
del contributions[105]
agg.finalize(agg.combine(contributions.values()))

## Members in parallel

`contribute` is a plain function of one row, so any executor can run it.
The aggregation does not own parallelism; whoever iterates over the members chooses how.
We warm the stages first, so that the threads do not each compute the held part:

In [ ]:
from concurrent.futures import ThreadPoolExecutor

sciline.warm(*agg.stages)
with ThreadPoolExecutor() as executor:
    combined = agg.combine(executor.map(agg.contribute, table.values()))
agg.finalize(combined)

## Per-member results without combining

To compute a key for each member without combining, use [compute_members](../generated/functions/sciline.compute_members.rst).
It returns the value for each row, by the row's label:

In [ ]:
sciline.compute_members(pipeline, members=(Filename,), key=Calibrated, table=table)

## A key that does not depend on the members

An accumulation key that does not depend on the member keys is not accumulated.
`finalize` computes it from the held part of the pipeline instead.
Here `Calibration` is requested as an accumulation key and as an output, but only `Total` is accumulated:

In [ ]:
agg = sciline.Aggregation(
    pipeline,
    members=(Filename,),
    accumulators={
        Total: sciline.Reduced(operator.add),
        Calibration: sciline.Reduced(operator.add),
    },
    outputs=(Result, Calibration),
)
agg.accumulation_keys

In [ ]:
agg.compute(table)

Whether a key is accumulated is decided by the graph, so a test should check `accumulation_keys` when the graph is refactored.

## Multiple member keys

A table may have several columns.
Here each run comes with its own calibration file, so the calibration is no longer held but computed per member:

In [ ]:
table = {
    102: {Filename: 'file102.txt', CalibrationFilename: 'calib_2025.txt'},
    103: {Filename: 'file103.txt', CalibrationFilename: 'calib_2025.txt'},
    104: {Filename: 'file104.txt', CalibrationFilename: 'calib_2026.txt'},
    105: {Filename: 'file105.txt', CalibrationFilename: 'calib_2026.txt'},
}
agg = sciline.Aggregation(
    pipeline,
    members=(Filename, CalibrationFilename),
    accumulators={Total: sciline.Reduced(operator.add)},
    outputs=(Result,),
)
agg.contribute_stage.frontier, agg.contribute_stage.dynamic

In [ ]:
agg.compute(table)